# Tobit 模型：归并、潜在净借款需求与边际效应

> 本讲义用于《金融数据分析与建模》课程。配套文件 `04_tobit_model_codes_v2.ipynb` 用于生成 `./figs/` 中的配图和 `./data/tobit_credit_sim.csv`。

本章围绕一个具体的金融问题展开：企业银行贷款金额为什么会有大量 0？Tobit 模型不是“只要因变量有很多 0 就可以用”的模型，而是建立在一个更具体的观测机制上：存在一个潜在连续变量 $B_i^*$，但实际观测到的贷款金额 $B_i$ 不能小于 0，因此所有 $B_i^*\le 0$ 的样本都被记录为 $B_i=0$。

本章把 `censoring` 主要译为“归并”。中文文献中也常见“删失”“审查”“截尾”等译法，但“截尾”容易与 `truncation` 混淆。因此，后文将 `censoring` 称为“归并”，将 `truncation` 称为“截断”。

## 1. 贷款金额为什么会出现大量 0？

考虑一组企业信贷数据。研究者观察到每家企业当年的银行贷款金额 $B_i$。有些企业有正的银行贷款，有些企业没有银行贷款，因此：

$$
B_i=0
$$

这种 0 可以有不同解释。本章先讨论 Tobit 适用的一种解释：企业有一个潜在净借款需求 $B_i^*$，实际贷款金额不能为负，所以当潜在净借款需求为负或不超过 0 时，数据中统一记录为 0。

这里的关键不是“贷款金额不能为负”本身，而是 $B_i^*$ 的经济含义。$B_i^*$ 不应被理解为实际贷款金额，而应理解为企业在权衡资金需求、投资机会、融资成本和内部资金之后形成的 **潜在净借款需求**。

## 2. 潜变量 $B_i^*$ 的经济含义

设企业 $i$ 的潜在净借款需求为：

$$
B_i^*
=
\alpha
+\beta_1 opportunity_i
+\beta_2 collateral_i
-\beta_3 cash_i
+u_i
$$

其中：

- $opportunity_i$ 表示企业投资机会或资金需求强度；
- $collateral_i$ 表示企业抵押能力或可抵押资产比例；
- $cash_i$ 表示内部资金充裕程度；
- $u_i$ 表示研究者没有观测到的净融资需求因素。

上述设定的含义是：投资机会越多，企业越可能需要外部融资；抵押能力越强，企业越容易获得银行贷款；内部现金越充裕，企业对外部贷款的净需求越弱。

实际观测到的银行贷款金额为：

$$
B_i=\max(0,B_i^*)
$$

也可以写成：

$$
B_i=
\begin{cases}
0, & B_i^*\le 0,\\
B_i^*, & B_i^*>0.
\end{cases}
$$

### 2.1 为什么 $B_i^*<0$ 不等于“负贷款”？

这是理解 Tobit 模型最关键的一步。$B_i^*<0$ 不表示企业向银行借了负数金额，而是表示企业处在“不使用银行贷款”的一侧。

例如，企业可能有以下情形：

- 投资机会较弱，新增项目的预期收益低于贷款成本；
- 内部现金流充裕，没有必要通过银行贷款补充资金；
- 企业把闲置资金用于理财、委托贷款或其他金融资产配置；
- 银行贷款的固定成本、审批成本或抵押约束使得企业没有使用贷款的净动机。

这些情形都可以使潜在净借款需求 $B_i^*$ 处于 0 以下。但由于本章的因变量是“银行贷款金额”，它不能为负。因此，所有 $B_i^*\le 0$ 的企业都会被记录为 $B_i=0$。这就是“归并”的含义：样本仍然在数据中，但边界外的潜在取值被统一归并到边界值。

![潜在净借款需求与观测贷款金额](./figs/limit_dep_tobit_fig01_latent_to_observed.png)

上图中，横轴是潜在净借款需求 $B_i^*$，纵轴是观测到的银行贷款金额 $B_i$。当 $B_i^*\le 0$ 时，$B_i$ 被归并为 0；当 $B_i^*>0$ 时，$B_i$ 等于潜在净借款需求。

In [1]:
import pandas as pd

df = pd.read_csv("./data/tobit_credit_sim.csv")
df[["loan_amt", "B_star", "opportunity", "collateral", "cash"]].describe().round(3)

,loan_amt,B_star,opportunity,collateral,cash
count,3000.000,3000.000,3000.000,3000.000,3000.000
mean,49.462,-0.277,0.001,0.000,0.000
std,76.179,1.847,1.007,1.000,1.000
min,0.000,-5.804,-3.888,-2.320,-1.798
25%,0.000,-1.579,-0.692,-0.768,-0.781
50%,0.000,-0.256,-0.001,0.032,-0.118
75%,84.326,1.054,0.685,0.783,0.659
max,424.339,5.304,4.184,2.084,3.752


## 3. 为什么不能直接用 OLS？

如果直接估计：

$$
B_i=x_i'\beta+\varepsilon_i
$$

OLS 会把 0 点堆积当作普通连续结果的一部分。问题在于，Tobit 的观测机制并不是普通线性误差，而是：

$$
B_i=
\begin{cases}
0, & B_i^*\le 0,\\
B_i^*, & B_i^*>0.
\end{cases}
$$

也就是说，$B_i=0$ 的样本贡献的是一个概率事件：

$$
P(B_i=0\mid x_i)=P(B_i^*\le 0\mid x_i)
$$

而 $B_i>0$ 的样本贡献的是连续密度。OLS 没有同时处理这两类信息，因此通常不能恢复潜在方程中的参数。

![潜在变量与观测变量分布](./figs/limit_dep_tobit_fig02_latent_observed_distribution.png)

图中可以看到，潜在净借款需求 $B_i^*$ 是连续变量；但经过归并以后，观测贷款金额 $B_i$ 在 0 点出现明显堆积。Tobit 的核心就是同时利用 0 点概率和正值部分的连续信息。

## 4. Tobit 模型的基本设定

标准左归并 Tobit 可以写成：

$$
B_i^*=x_i'\beta+u_i,\quad u_i\mid x_i\sim N(0,\sigma^2)
$$

$$
B_i=\max(0,B_i^*)
$$

在本章案例中，主解释变量设为：

$$
x_i=(1,\ opportunity_i,\ collateral_i,\ cash_i)
$$

这一组变量共同影响潜在净借款需求，因此也共同影响两个结果：

- 企业是否有正的银行贷款；
- 在贷款金额为正时，贷款金额有多大。

这就是 Tobit 与 Two-part model 的根本区别。Tobit 假定是否贷款和贷多少来自同一个潜在连续过程；Two-part model 则允许这两个机制分开。

![x 如何影响潜在净借款需求](./figs/limit_dep_tobit_fig03_x_to_latent_demand.png)

在 Tobit 模型中，同一组 $x$ 通过潜在净借款需求 $B_i^*$ 同时影响正贷款概率和正值金额。抵押能力提高时，整条潜在需求曲线上移，因此企业更可能出现正贷款，且正贷款金额也会更大。

## 5. Tobit 的似然函数

对于 $B_i=0$ 的样本，似然贡献是：

$$
P(B_i=0\mid x_i)
=
P(B_i^*\le 0\mid x_i)
=
\Phi\left(-\frac{x_i'\beta}{\sigma}\right)
$$

对于 $B_i>0$ 的样本，观测到 $B_i=B_i^*$，其密度为：

$$
f(B_i\mid x_i)
=
\frac{1}{\sigma}
\phi\left(
\frac{B_i-x_i'\beta}{\sigma}
\right)
$$

因此，对数似然函数为：

$$
\ell(\beta,\sigma)
=
\sum_{B_i=0}
\log
\Phi\left(-\frac{x_i'\beta}{\sigma}\right)
+
\sum_{B_i>0}
\left[
-\log\sigma+
\log\phi\left(\frac{B_i-x_i'\beta}{\sigma}\right)
\right]
$$

这也是 Tobit 与 OLS 的根本差别：Tobit 明确把 0 点样本作为概率事件处理。

## 6. Tobit 的边际效应

Tobit 模型中的系数 $\beta_j$ 不是一个单一边际效应。因为 $x_j$ 同时影响：

- 正贷款的概率；
- 无条件期望贷款金额；
- 正贷款样本中的条件均值。

设：

$$
z_i=\frac{x_i'\beta}{\sigma}
$$

则正值概率为：

$$
P(B_i>0\mid x_i)=\Phi(z_i)
$$

非条件期望为：

$$
E(B_i\mid x_i)
=
\Phi(z_i)x_i'\beta+\sigma\phi(z_i)
$$

因此，对非条件期望的边际效应为：

$$
\frac{\partial E(B_i\mid x_i)}{\partial x_{ij}}
=
\Phi(z_i)\beta_j
$$

对正贷款概率的边际效应为：

$$
\frac{\partial P(B_i>0\mid x_i)}{\partial x_{ij}}
=
\phi(z_i)\frac{\beta_j}{\sigma}
$$

正值条件均值为：

$$
E(B_i\mid B_i>0,x_i)
=
x_i'\beta+\sigma\lambda(z_i)
$$

其中：

$$
\lambda(z_i)=\frac{\phi(z_i)}{\Phi(z_i)}
$$

其边际效应为：

$$
\frac{\partial E(B_i\mid B_i>0,x_i)}{\partial x_{ij}}
=
\beta_j
\left[
1-\lambda(z_i)\{z_i+\lambda(z_i)\}
\right]
$$

![Tobit 的三类边际效应](./figs/limit_dep_tobit_fig04_marginal_effects.png)

图中展示的是 `opportunity` 的三类边际效应。对于 Tobit 模型，不能只看 $\beta_j$ 后直接说“贷款金额增加多少”。更准确的做法是说明它影响的是哪一个对象：正贷款概率、非条件期望，还是正值条件均值。

## 7. 与 Two-part model 的关系

Tobit 和 Two-part model 都可以处理贷款金额中大量 0 的问题，但二者的经济含义不同。

| 问题 | Tobit | Two-part model |
|---|---|---|
| 0 的含义 | 潜在净借款需求被归并到 0 | 真实的无贷款状态 |
| 机制数量 | 一个潜在连续机制 | 是否贷款与贷多少两个机制 |
| 变量作用 | 同一组 $x$ 同时影响概率和金额 | 不同变量可以分别进入两个方程 |
| 适合场景 | 共同机制假设较合理 | 参与决策与强度决策可分离 |

在实证研究中，二者不是机械的替代关系。关键是研究者如何理解 $B_i=0$ 的生成机制。如果 0 是潜在净借款需求被归并后的结果，Tobit 是自然起点；如果 0 是真实的不参与状态，并且是否贷款与贷多少有不同决定因素，则 Two-part model 更自然。

## 8. 小结

本章的核心不是记住 Tobit 的命令，而是理解三个问题：

- $B_i^*$ 是潜在净借款需求，不是实际贷款金额；
- $B_i^*<0$ 表示企业处在不使用银行贷款的一侧，不表示负贷款；
- Tobit 假定同一组 $x$ 通过同一个潜在机制同时影响 $B_i=0$ 的概率和 $B_i>0$ 时的金额。

只要这三个问题讲清楚，后续的 Two-part model 和 Heckman selection 就会更容易理解。